# The Vibe Tax — LiveCodeBench pipeline (end to end)

Run the whole functional-code-evaluation pipeline in one place: **problems → tests → prompts → model responses → extraction & scoring → analysis.** The code-extraction step (`score_lcb.py :: extract_solution`) is the subject of the extraction-artifacts paper; this notebook lets you see it and run it against real data.

**How to use in Colab**
1. `Runtime → Run all`, or run cells top to bottom.
2. Optional secrets (`🔑` in the left sidebar → *Secrets*, or you'll be prompted): `HF_TOKEN` (to rebuild LCB tests), `OPENAI_API_KEY` / `ANTHROPIC_API_KEY` / `CODESTRAL_API_KEY` (only if you re-query models), `GITHUB_TOKEN` (only if the repo is private).
3. **By default nothing costs money** — the model responses are already committed, so scoring re-runs on them. The API stage is off behind a flag.

> The only file the repo does **not** ship is `lcb_tests.jsonl` (LCB tests are HF-gated). The notebook rebuilds it from HuggingFace (needs `HF_TOKEN` + dataset access), or you can upload your local copy.

## Data-flow graph (file → script → file)

```
                         [HuggingFace: livecodebench/code_generation_lite]
                                    |                      |
                    prepare_lcb.py  |                      |  extract_lcb_tests.py
                                    v                      v
                          lcb_problems.jsonl         lcb_tests.jsonl        (both LOCAL: HF-gated)
                                    |                      |
              generate_lcb_prompts.py                      |
                                    v                      |
                          lcb_v3_prompts.json              |
                                    |                      |
                       run_vibe_tax.py  (API calls)        |
                                    v                      |
                        lcb_v3_responses.json              |
                                    |                      |
                                    +----------+-----------+
                                               v
                                        score_lcb.py   <-- the extractor lives here
                                               v
                          lcb_scored.json  +  lcb_scored_stats.json
                                               |
                    +--------------------------+---------------------------+
                    v                          v                           v
             mcnemar_lcb.py         full_partial_credit.py        analyze_lcb_capability.py
                    v                          v                           v
        lcb_mcnemar_stats.json   full_partial_credit.json      lcb_capability_stats.json
                                               |
                                               v
                                   build_breakdown.py  (repo root)
                                               v
                              The_Vibe_Tax_Problem_Breakdown.xlsx
```

## Stage table

| # | Script | Reads | Writes | Needs | In this notebook |
|---|--------|-------|--------|-------|------------------|
| 1 | `prepare_lcb.py` | HuggingFace LCB | `lcb_problems.jsonl` | HF_TOKEN | **committed** — optional rebuild |
| 2 | `extract_lcb_tests.py` | HuggingFace LCB | `lcb_tests.jsonl` | HF_TOKEN | rebuilt (gitignored) or upload |
| 3 | `generate_lcb_prompts.py` | `lcb_problems.jsonl` | `lcb_v3_prompts.json` | — (mock backend) | rebuilt, keyless |
| 4 | `run_vibe_tax.py` | `lcb_v3_prompts.json` | `lcb_v3_responses.json` | API keys ($) | **committed** — off by default |
| 5 | `score_lcb.py` | tests + responses + problems | `lcb_scored.json` (+stats) | — | **runs the extractor** |
| 6a | `mcnemar_lcb.py` | `lcb_scored.json` | `lcb_mcnemar_stats.json` | — | runs |
| 6b | `full_partial_credit.py` | responses + tests | `full_partial_credit.json` | — (heavy) | optional |
| 6c | `analyze_lcb_capability.py` | `lcb_scored.json` | `lcb_capability_stats.json` | — | optional |
| 6d | `build_breakdown.py` | scored + responses + problems | `.xlsx` workbook | — | optional |

### 0 · Install dependencies

In [ ]:
!pip -q install openai anthropic huggingface_hub openpyxl

### 0.1 · Clone the repo and enter the pipeline directory
If the repo is private, add a `GITHUB_TOKEN` secret (a GitHub PAT with `repo` read).

In [ ]:
import os
BRANCH = "claude/blissful-hopper-dy0rvr"
REPO   = "jamelski1/the-vibe-tax"

tok = None
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")   # optional; only for a private repo
except Exception:
    tok = os.environ.get("GITHUB_TOKEN")

url = f"https://{tok}@github.com/{REPO}.git" if tok else f"https://github.com/{REPO}.git"
if not os.path.isdir("the-vibe-tax"):
    !git clone --quiet --branch {BRANCH} {url}
else:
    print("repo already cloned")
%cd the-vibe-tax/data/vibe_tax_lcb
import sys; sys.path.insert(0, ".")
!git log --oneline -1

### 0.2 · Configure secrets (optional)
Pulls from Colab *Secrets* if present, otherwise prompts. Skip any you don't need — scoring the committed responses needs **none** of these.

In [ ]:
import os, getpass
def _secret(name, prompt_if_missing=False):
    v = None
    try:
        from google.colab import userdata
        v = userdata.get(name)
    except Exception:
        v = os.environ.get(name)
    if not v and prompt_if_missing:
        v = getpass.getpass(f"{name} (leave blank to skip): ") or None
    if v:
        os.environ[name] = v
    return v

have = {k: bool(_secret(k)) for k in
        ("HF_TOKEN","OPENAI_API_KEY","ANTHROPIC_API_KEY","CODESTRAL_API_KEY")}
print("secrets present:", have)
print("\nNothing above is required to score the committed responses (stage 5).")

### Stage 1 · `prepare_lcb.py` → `lcb_problems.jsonl`  *(optional — already committed)*
The 167-problem file ships with the repo. Only rebuild it if you want to change the filters (date window, difficulties). Needs `HF_TOKEN` + accepting the dataset terms on HuggingFace once.

In [ ]:
RUN_PREPARE = False   # flip to True to rebuild the problem set from HuggingFace

import os
if RUN_PREPARE:
    !python prepare_lcb.py --min-date 2024-08-01 --difficulties easy,medium,hard --limit 0
else:
    n = sum(1 for _ in open("lcb_problems.jsonl"))
    print(f"using committed lcb_problems.jsonl ({n} problems)")

### Stage 2 · `extract_lcb_tests.py` → `lcb_tests.jsonl`  *(required for scoring)*
LCB's graded tests are HF-gated and **not** committed. This rebuilds them (needs `HF_TOKEN`). If HF isn't available, upload your local `lcb_tests.jsonl` with the commented `files.upload()` block.

In [ ]:
import os
if os.path.exists("lcb_tests.jsonl"):
    print("lcb_tests.jsonl already present")
elif os.environ.get("HF_TOKEN"):
    !python extract_lcb_tests.py
else:
    print("No HF_TOKEN and no lcb_tests.jsonl.")
    print("Option A: add an HF_TOKEN secret and re-run this cell.")
    print("Option B: upload your local file by uncommenting below:")
    # from google.colab import files
    # up = files.upload()            # choose your lcb_tests.jsonl
    # open("lcb_tests.jsonl","wb").write(next(iter(up.values())))

print("tests ready:" , os.path.exists("lcb_tests.jsonl"))

### Stage 3 · `generate_lcb_prompts.py` → `lcb_v3_prompts.json`
Wraps each problem in the 4 framings (terse / casual / detailed / multilingual). The **mock** backend is deterministic and needs no API key.

In [ ]:
!python generate_lcb_prompts.py --backend mock

import json
prompts = json.load(open("lcb_v3_prompts.json"))
print(f"{len(prompts)} prompts (167 problems x 4 framings)\n")
print("example entry:")
e = prompts[0]
print(" task_id:", e["task_id"], "| condition:", e["condition"])
print(" prompt (first 400 chars):\n", e["prompt"][:400], "...")

### The heart of it · the extractor code
`score_lcb.py` turns a free-form model reply into a runnable program. These are the functions the paper is about — read them here before running them.

In [ ]:
import inspect, score_lcb
for fn in (score_lcb.extract_solution, score_lcb._trim_to_compilable,
           score_lcb.resolve_callable, score_lcb.passes):
    print("="*78)
    print(inspect.getsource(fn))

### Stage 4 · `run_vibe_tax.py` → `lcb_v3_responses.json`  *(OPTIONAL — costs money)*
The model responses are **already committed**, so leave this off to score them as-is. Flip `RUN_API = True` only to re-query (needs API keys and spends tokens). It resumes from a progress file, so a partial run is safe.

In [ ]:
RUN_API = False   # True re-queries the models ($$$). Default: use committed responses.

import os
if RUN_API:
    os.environ["PROMPTS_FILE"]  = os.path.abspath("lcb_v3_prompts.json")
    os.environ["OUTPUT_FILE"]   = os.path.abspath("lcb_v3_responses.json")
    os.environ["PROGRESS_FILE"] = os.path.abspath("lcb_v3_progress.json")
    # os.environ["ONLY_MODELS"] = "chatgpt"   # optional: one provider
    !python ../vibe_tax_v2/run_vibe_tax.py
else:
    import json
    r = json.load(open("lcb_v3_responses.json"))
    print(f"using committed lcb_v3_responses.json ({len(r)} completions)")

### Stage 5 · `score_lcb.py` → `lcb_scored.json`  ← **the extractor runs here**
For every completion: `extract_solution()` pulls runnable code, `resolve_callable()` binds the entry point in any shape, and `passes()` runs each test under a per-test timeout. Writes a `passed` bool per record plus aggregate stats.

In [ ]:
import os, json
if not os.path.exists("lcb_tests.jsonl"):
    print("Cannot score without lcb_tests.jsonl — run Stage 2 first.")
else:
    !python score_lcb.py --max-tests 60
    stats = json.load(open("lcb_scored_stats.json"))
    print("\nOVERALL: {pass_rate}%  (n={total})".format(**stats["overall"]))
    print("by condition:")
    for k,v in stats["by_condition"].items():
        print(f"  {k:22s} {v['pass_rate']}%")
    print("by difficulty:")
    for k,v in stats["by_difficulty"].items():
        print(f"  {k:8s} {v['pass_rate']}%")

### Stage 6a · `mcnemar_lcb.py` → `lcb_mcnemar_stats.json`
Paired within-problem test of terse vs each other framing (the framing-null result).

In [ ]:
import json
!python mcnemar_lcb.py --base agentic_terse
m = json.load(open("lcb_mcnemar_stats.json"))
print("\nterse vs each framing (capable-models slice):")
for pair, slices in m.items():
    cm = slices.get("capable_models")
    if cm:
        print(f"  {pair:44s} delta={cm['delta_pts']:+.1f} pts  p={cm['p']}  "
              f"{'sig' if cm['significant'] else 'n.s.'}")

### Stage 6b–d · deeper analyses  *(optional; some are heavy)*
Partial credit re-runs failed attempts (slow), capability slices by topic, and the Excel workbook.

In [ ]:
RUN_PARTIAL   = False   # slow: re-runs every failed attempt for per-test %
RUN_CAPABILITY = True
RUN_WORKBOOK   = True

if RUN_PARTIAL:
    !python full_partial_credit.py --timeout 6
if RUN_CAPABILITY:
    !python analyze_lcb_capability.py 2>/dev/null || echo "(analyze_lcb_capability.py needs lcb_scored.json)"
if RUN_WORKBOOK:
    !python ../../build_breakdown.py
    print("\nWorkbook written to repo root: The_Vibe_Tax_Problem_Breakdown.xlsx")
    # from google.colab import files; files.download("../../The_Vibe_Tax_Problem_Breakdown.xlsx")

### Inspect / debug side-paths (no full run)
- `print_prompt.py <task_id>` — the exact prompt that would be sent.
- `manual_test.py` — paste a reply, run `extract_solution` + tests on that one completion (the tool that surfaced the extractor bugs).
- `test_class_shape.py` — unit-tests the shape resolver and scans the corpus for shape-mismatch prevalence.

In [ ]:
# see the extractor on a single completion, in isolation:
import json, score_lcb
r = json.load(open("lcb_v3_responses.json"))[0]
code_out = score_lcb.extract_solution(r["completion"], r["entry_point"])
print("task:", r["task_id"], "| entry:", r["entry_point"])
print("extracted code (first 300 chars):\n", (code_out or "")[:300])

print("\n--- shape-resolver unit test + corpus prevalence scan ---")
!python test_class_shape.py

---
**Recap.** Stages 1–4 build the inputs (problems, tests, prompts, responses); Stage 5 is where the extractor turns replies into pass/fail; Stage 6 analyses the scores. Re-scoring is decoupled from the API — once `lcb_v3_responses.json` exists, changing the extractor and re-running **Stage 5 only** (no model calls) is how every fix in the paper was validated.